# BraTS2023-GLI — Phân đoạn u não 3D U-Net (Colab T4)

**Bản V2 ổn định:** Đường dẫn Drive được đối chiếu với cache `.npz` đã có; kiểm tra đầy đủ dữ liệu; chia tập theo patient; sử dụng 3D U-Net + MONAI; **checkpoint từng 25 batch** để tiếp tục sau khi Colab ngắt; xác thực train/val/test; thống kê Dice, IoU, HD95; hiển thị phân đoạn và xuất báo cáo nghiên cứu.

**Cách chạy:** Thêm lối tắt `BraTS2023_GLI` từ *Được chia sẻ với tôi* vào *Drive của tôi* → chọn GPU T4 → `Chạy tất cả`. Huấn luyện có thể kéo dài qua nhiều phiên Colab. Nếu ngắt, chạy lại từ đầu notebook và train sẽ phục hồi bước đã lưu. Nếu train chưa hoàn tất, phần kết quả cuối tự bỏ qua, **không báo RuntimeError giả**.

> **Không tạo số liệu giả.** Checkpoint V2 dùng `EXPERIMENT_ID` riêng và không đụng vào các checkpoint V1. Chỉ đọc dữ liệu `.npz` đã kiểm định tương thích; nếu định dạng khác sẽ báo nguyên nhân, không gộp bừa. Mục đích nghiên cứu, **không dùng chẩn đoán y khoa**.

## 1. Cài đặt các thư viện phụ thuộc

In [ ]:
# MONAI/nibabel/FPDF: use existing Colab GPU PyTorch; do not reinstall torch/CUDA.
%pip -q install "monai>=1.5,<1.7" "nibabel>=5.2,<6" "fpdf2>=2.7,<3" "scipy>=1.11,<2" "scikit-learn>=1.3,<2"


In [ ]:
import gc
import csv
import zipfile
import platform
import hashlib
import uuid
import json
import re
import os
import random
import shutil
import time
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
from zipfile import BadZipFile
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
from scipy.ndimage import binary_erosion, distance_transform_edt
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from scipy.ndimage import label as nd_label
import monai.transforms as mt
from monai.data import DataLoader as MonaiDataLoader
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.networks.nets import UNet
from monai.utils import set_determinism


## 2. Mount Google Drive và Thiết lập Thiết bị

In [ ]:
# Mount Google Drive only when running in Colab. Check MyDrive, not the mount directory alone.
IN_COLAB = False
try:
    from google.colab import drive

    IN_COLAB = True
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive", force_remount=False)
    if not Path("/content/drive/MyDrive").exists():
        raise RuntimeError("Google Drive mount failed: /content/drive/MyDrive is unavailable")
    print("Google Drive is ready")
except ImportError:
    print("Running outside Google Colab")

In [ ]:
# Reproducibility across Python, NumPy, PyTorch, MONAI transforms and DataLoader workers.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_determinism(seed=SEED)
torch.backends.cudnn.benchmark = False # tắt chế độ benchmark (chế độ này chọn thuật toán tối ưu nhưng có thể không tái lập).
torch.backends.cudnn.deterministic = True # buộc các phép toán dùng thuật toán deterministic (kết quả giống nhau mỗi lần chạy).

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / (1024 ** 3):.2f} GB")

### Đã sửa lỗi đường dẫn NPZ và kiểm tra trước khi train

**Cách chạy:** Mount Drive → `V1_CONFIG` → tiền xử lý/kiểm tra cache → split → Dataset/DataLoader → `V1_TRAIN`. Với dữ liệu ở **Được chia sẻ với tôi**, thêm **lối tắt** thư mục `BraTS2023_GLI` vào **Drive của tôi** trước khi chạy. Nếu lối tắt có tên khác, sửa `DATASET_DIR` ở `V1_CONFIG`.

Notebook ưu tiên cache cũ `processed_npz_v1_brats2023`, đồng thời tìm cache thay thế ở `Capstone_Training/processed_npz_strict_v1/brats2023`. Cache thay thế **chỉ được dùng nếu đúng cấu trúc và phiên bản tiền xử lý**; không tự đổi nhãn, bỏ qua ca lỗi hay làm mới checkpoint. Khi thiếu cache gốc nhưng còn NIfTI tương ứng, notebook có thể tạo lại ca bị thiếu. Nếu thiếu cả hai, chương trình dừng với tên file cụ thể.

**Checkpoint:** Giữ `EXPERIMENT_ID='unet_v1_t4_001'`, `RESUME_V1=True`. Nếu fingerprint dữ liệu hoặc protocol khác checkpoint, chương trình sẽ dừng để tránh resume sai mô hình. Chưa thể kiểm chứng dữ liệu trên Drive từ môi trường này; cần chạy các cell trên Colab để xác nhận.


In [ ]:
# CELL: V2_CONFIG
# Chỉ sửa cấu hình trước khi bắt đầu một EXPERIMENT_ID mới.
DATASET_DIR = Path('/content/drive/MyDrive/BraTS2023_GLI') if IN_COLAB else Path('data/BraTS2023_GLI')
# In Drive > Shared with me, add a SHORTCUT of BraTS2023_GLI to My Drive.
# If your shortcut has a different name, edit DATASET_DIR above.
# Optional: set CACHE_DIR_OVERRIDE to an exact mounted directory containing .npz files.
CACHE_DIR_OVERRIDE = None
CACHE_FALLBACK_DIRS = [
    DATASET_DIR / 'Capstone_Training' / 'processed_npz_strict_v1' / 'brats2023',
    DATASET_DIR / 'processed_npz_strict_v1' / 'brats2023',
]

RAW_DIR_OVERRIDE = None  # Có thể đặt Path('...') tới thư mục chứa trực tiếp các ca BraTS-GLI-...
EXPERIMENT_ID = 'unet_v2_resumable_001'  # New run; original v1 checkpoints are not overwritten
RAW_LABEL_SCHEMA = 'brats2023'  # ET=3; dùng 'legacy4' chỉ khi ET=4.
if RAW_LABEL_SCHEMA not in ('brats2023', 'legacy4'):
    raise ValueError('RAW_LABEL_SCHEMA must be brats2023 or legacy4')
if not re.fullmatch(r'[A-Za-z0-9_-]+', EXPERIMENT_ID):
    raise ValueError('EXPERIMENT_ID may contain only letters, digits, _ and -')

def is_brats_case_name(name: str) -> bool:
    return re.fullmatch(r'BraTS-[A-Za-z]+-\d+-\d+', name) is not None


def resolve_raw_data(dataset_dir: Path, cache_dir: Path,
                     raw_override: Optional[Path] = None,
                     cache_candidates=()) -> Optional[Path]:
    """Find one cohort inside the specified dataset folder; never scan all of Drive."""
    dataset_dir, cache_dir = Path(dataset_dir), Path(cache_dir)
    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f'Dataset folder is not visible: {dataset_dir}\n'
            'If BraTS2023_GLI is in Shared with me, add its shortcut to My Drive '
            'with the name BraTS2023_GLI, then rerun the Drive mount and configuration cells. '
            'Opening the notebook from a shared folder does not mount that folder as MyDrive. '
            'Otherwise set DATASET_DIR to the actual mounted folder. '
            'No empty dataset folder has been created.')

    def children(folder):
        try:
            return sorted(folder.iterdir(), key=lambda p: p.name)
        except PermissionError as exc:
            raise PermissionError(f'Cannot read {folder}. Check the shared folder access.') from exc

    def contains_cases(folder):
        return any(is_brats_case_name(p.name) and p.is_dir() for p in children(folder))

    if raw_override is not None:
        override = Path(raw_override)
        if not override.is_dir() or not contains_cases(override):
            raise ValueError(f'RAW_DIR_OVERRIDE must directly contain BraTS case folders: {override}')
        return override

    # Detect direct cases and at most two wrapper directories, including duplicate extraction names.
    queue, found, visited = [(dataset_dir, 0)], [], set()
    ignored = ('processed_npz', 'runs', 'checkpoints', 'reports', 'predictions', 'models',
               'outputs', 'scripts', 'web', 'app', 'src', 'tests', '__MACOSX')
    while queue:
        folder, depth = queue.pop(0)
        canonical = folder.resolve()
        if canonical in visited:
            continue
        visited.add(canonical)
        if len(visited) > 100:
            raise RuntimeError('Dataset search exceeded 100 folders. Set RAW_DIR_OVERRIDE to the case-parent folder.')
        entries = children(folder)
        if any(is_brats_case_name(p.name) and p.is_dir() for p in entries):
            found.append(folder)
            continue  # Never inspect all MRI files in every patient directory here.
        if depth < 2:
            for entry in entries:
                if (not entry.name.startswith('.') and not entry.name.startswith(ignored)
                        and entry.is_dir()):
                    queue.append((entry, depth + 1))
    if len(found) > 1:
        choices = '\n'.join(str(p) for p in found)
        raise ValueError(f'Multiple MRI cohorts found; set RAW_DIR_OVERRIDE explicitly:\n{choices}')
    if found:
        return found[0]
    available_cache = next(
        (p for p in (cache_dir, *cache_candidates) if p.is_dir() and any(p.glob('*.npz'))),
        None,
    )
    if available_cache is not None:
        print(f'Raw MRI not found; using existing NPZ caches from {available_cache}.')
        print('The cache metadata and preprocessing version will be checked before training.')
        return None
    names = ', '.join(p.name for p in children(dataset_dir)) or '(empty folder)'
    raise FileNotFoundError(
        f'No BraTS case folders or current NPZ caches found under {dataset_dir}.\n'
        f'Visible items: {names}\n'
        f'Expected cache: {cache_dir}\n'
        'A folder containing only notebooks is not MRI training data. '
        'If MRI data is still in a .zip/.tar archive, extract it first. '
        'For deeper nesting or data stored elsewhere, set RAW_DIR_OVERRIDE to the '
        'folder directly containing BraTS-GLI-XXXXX-XXX cases.')


# The original canonical cache stays the preferred destination for any rebuilt cases.
npz_dir = (Path(CACHE_DIR_OVERRIDE) if CACHE_DIR_OVERRIDE is not None
           else DATASET_DIR / f'processed_npz_v1_{RAW_LABEL_SCHEMA}')
CACHE_FALLBACK_DIRS = [p for p in CACHE_FALLBACK_DIRS if p != npz_dir]
raw_dir = resolve_raw_data(
    DATASET_DIR, npz_dir, RAW_DIR_OVERRIDE,
    cache_candidates=CACHE_FALLBACK_DIRS,
)
data_root = raw_dir

drive_npz_dir = npz_dir
local_npz_dir = Path('/content') / npz_dir.name if IN_COLAB else npz_dir
run_dir = DATASET_DIR / 'runs_v2' / EXPERIMENT_ID
checkpoints_dir = run_dir / 'checkpoints'
reports_dir = run_dir / 'reports'
predictions_dir = run_dir / 'predictions'
export_dir = run_dir / 'demo_bundle'
for output_dir in (checkpoints_dir, reports_dir, predictions_dir, export_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
PREPROCESS_VERSION = f'brats_npz_v5_{RAW_LABEL_SCHEMA}'
FORCE_REBUILD_CACHE = False
REVALIDATE_CACHE = False  # True sau khi thay nội dung dữ liệu hoặc cần kiểm định lại.
LOCAL_CACHE_GB = 8.0  # Không chép toàn bộ cohort lên ổ đĩa Colab.
MODALITIES = ('t1n', 't1c', 't2w', 't2f')
LABEL_MAP = {0: 0, 1: 1, 2: 2, (3 if RAW_LABEL_SCHEMA == 'brats2023' else 4): 3}
ALLOWED_RAW_LABELS = set(LABEL_MAP)
VAL_SPLIT, TEST_SPLIT = 0.15, 0.15
VAL_SUBSET_SIZE = 20
PATCH_SIZE = (96, 96, 96)
BATCH_SIZE, NUM_SAMPLES = 1, 2  # Hai patch 96³ / ca, phù hợp bộ nhớ T4.
NUM_WORKERS = 0  # Required for deterministic mid-epoch resume / safe disk cache
STEPS_PER_EPOCH = None  # None = hết tập train. Đặt số chỉ cho thử nghiệm có giới hạn.
MAX_EPOCHS = 50
EPOCHS_PER_SESSION = 50  # One click tries all epochs; resume after runtime interruption
FAST_VAL_INTERVAL, FULL_VAL_INTERVAL = 1, 5
EARLY_STOP_PATIENCE = 6  # Số lần full validation liên tiếp, không phải số epoch.
EARLY_STOP_MIN_DELTA = 0.001
SW_BATCH_SIZE, SW_OVERLAP = 1, 0.5
INFERENCE_USE_TTA = False  # Cố định trước run; True = thêm flip-X, tốn ~2 lần inference.
SMALL_COMPONENT_VOXELS = 50
LR, WEIGHT_DECAY = 2e-4, 1e-4
MODELS_TO_TRAIN = ['unet3d']
RESUME_V1 = True  # Resume last completed mini-batch checkpoint if available.
SAVE_EVERY_STEPS = 25  # Save complete model/optimizer/scaler state after 25 training batches
SPLIT_MANIFEST_PATH = reports_dir / 'split_manifest.json'
METRIC_POLICY = 'custom_pooled_surface_hd95_26connectivity_empty_diagonal_v1'
print(f'Experiment: {run_dir}')
print(f'Raw data: {data_root}')
print(f'Primary cache: {drive_npz_dir}; local cache limit: {LOCAL_CACHE_GB:g} GB')
for backup_cache in CACHE_FALLBACK_DIRS:
    if backup_cache.is_dir():
        print(f'Alternate cache: {backup_cache} ({sum(1 for _ in backup_cache.glob("*.npz"))} NPZ)')
print('A different cache version is NOT automatically treated as compatible.')
print(f'3D U-Net | {MAX_EPOCHS} epochs | checkpoint each {SAVE_EVERY_STEPS} batches')
if device.type != 'cuda':
    print('WARNING: CPU detected. Chọn Runtime > Change runtime type > T4 GPU trước khi train.')


## 3. Tiền xử lý dữ liệu sang định dạng NPZ

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

def plot_eda_from_raw_nifti(data_root, num_samples=5):
    """
    Đọc và trực quan hóa 5 ca bệnh mẫu trực tiếp từ các file NIfTI thô (.nii.gz).
    """
    # 1. Lấy danh sách các thư mục ca bệnh raw
    raw_cases = sorted([d.name for d in data_root.iterdir() if d.is_dir() and is_brats_case_name(d.name)])[:num_samples]

    # 2. Bảng màu nhãn u chuẩn (0: Nền trong suốt, 1: Đỏ, 2: Xanh lá, 3: Vàng cam)
    if not raw_cases:
        raise ValueError('No raw cases available for EDA')
    num_samples = len(raw_cases)
    segmentation_cmap = ListedColormap(["none", "#e41a1c", "#4daf4a", "#ff7f00"])

    fig, axes = plt.subplots(num_samples, 5, figsize=(20, 4 * num_samples), facecolor="white")
    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    modality_suffixes = ["t1n", "t1c", "t2w", "t2f"]
    modality_names = ["T1 Native (t1n)", "T1 Contrast (t1c)", "T2 Weighted (t2w)", "T2 FLAIR (t2f)"]

    for row_idx, case_id in enumerate(raw_cases):
        case_folder = data_root / case_id

        # Đọc file segmentation nhãn u
        seg_path = case_folder / f"{case_id}-seg.nii.gz"
        seg_data = nib.load(seg_path).get_fdata()

        unexpected = set(np.unique(seg_data)) - ALLOWED_RAW_LABELS
        if unexpected:
            raise ValueError(f'{case_id}: labels {unexpected} do not match {RAW_LABEL_SCHEMA}')
        label_mapped = np.zeros_like(seg_data)
        for raw_label, internal_label in LABEL_MAP.items():
            label_mapped[seg_data == raw_label] = internal_label

        # Tìm lát cắt Axial Z có diện tích u lớn nhất
        z_sum = np.sum(label_mapped > 0, axis=(0, 1))
        z_idx = int(np.argmax(z_sum)) if z_sum.max() > 0 else seg_data.shape[2] // 2

        # Đọc và vẽ 4 file chuỗi xung MRI gốc
        flair_slice = None
        for col_idx, mod in enumerate(modality_suffixes):
            nii_path = case_folder / f"{case_id}-{mod}.nii.gz"
            nii_img = nib.load(nii_path).get_fdata()
            slice_data = nii_img[:, :, z_idx]

            if mod == "t2f":
                flair_slice = slice_data

            ax = axes[row_idx, col_idx]
            ax.imshow(slice_data.T, cmap="gray", origin="lower")
            if row_idx == 0:
                ax.set_title(modality_names[col_idx], fontsize=13, fontweight="bold", pad=8)
            if col_idx == 0:
                ax.set_ylabel(f"{case_id}\n(Slice Z: {z_idx+1})", fontsize=11, fontweight="bold")
            ax.axis("off")

        # Cột 5: Vẽ ảnh FLAIR đè nhãn Ground Truth Segmentation
        ax_seg = axes[row_idx, 4]
        mask_slice = label_mapped[:, :, z_idx].astype(float)
        mask_slice[mask_slice == 0] = np.nan

        ax_seg.imshow(flair_slice.T, cmap="gray", origin="lower")
        ax_seg.imshow(mask_slice.T, cmap=segmentation_cmap, vmin=0, vmax=3, alpha=0.6, origin="lower")
        if row_idx == 0:
            ax_seg.set_title("Raw Ground Truth Overlay", fontsize=13, fontweight="bold", pad=8)
        ax_seg.axis("off")

    # Legend
    legend_patches = [
        mpatches.Patch(color="#e41a1c", label="NCR - Lõi hoại tử"),
        mpatches.Patch(color="#4daf4a", label="ED - Vùng phù nề"),
        mpatches.Patch(color="#ff7f00", label="ET - U bắt thuốc cản quang"),
    ]
    fig.legend(handles=legend_patches, loc="upper center", bbox_to_anchor=(0.5, 0.99),
               ncol=3, fontsize=12, frameon=True, facecolor="#f8fafc", edgecolor="#cbd5e1")

    plt.suptitle("EDA: Khảo sát 5 ca bệnh từ tệp tin gốc NIfTI (.nii.gz)",
                 fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

# Chạy EDA trực tiếp từ thư mục raw
if data_root is not None:
    plot_eda_from_raw_nifti(data_root, num_samples=5)
else:
    print('Raw NIfTI unavailable; skipping raw-data EDA.')


In [ ]:
CACHE_ERRORS = (OSError, ValueError, TypeError, KeyError, EOFError, BadZipFile)
REQUIRED_CACHE_KEYS = {'image', 'label', 'affine', 'spacing', 'preprocess_version', 'case_id', 'modalities'}


def load_validated_cache(npz_path: Path) -> Dict[str, np.ndarray]:
    """Require an explicit labeled cache contract; never invent labels or geometry."""
    npz_path = Path(npz_path)
    with np.load(npz_path, allow_pickle=False) as data:
        missing = REQUIRED_CACHE_KEYS - set(data.files)
        if missing:
            raise ValueError(f'{npz_path}: missing {sorted(missing)}; rebuild from raw NIfTI')
        if str(data['preprocess_version'].item()) != PREPROCESS_VERSION:
            raise ValueError(f'{npz_path}: outdated preprocessing; rebuild from raw NIfTI')
        if str(data['case_id'].item()) != npz_path.stem:
            raise ValueError(f'{npz_path}: case_id/filename mismatch')
        if tuple(data['modalities'].tolist()) != MODALITIES:
            raise ValueError(f'{npz_path}: modality order mismatch')
        image = np.asarray(data['image'])
        label = np.asarray(data['label'])
        affine = np.asarray(data['affine'], dtype=np.float64)
        spacing = np.asarray(data['spacing'], dtype=np.float64)
    if image.ndim != 4 or image.shape[0] != 4 or any(size <= 0 for size in image.shape[1:]):
        raise ValueError(f'{npz_path}: expected image (4,H,W,D), got {image.shape}')
    if label.shape != (1,) + image.shape[1:]:
        raise ValueError(f'{npz_path}: label/image shape mismatch')
    if not np.isfinite(image).all() or not np.isfinite(label).all():
        raise ValueError(f'{npz_path}: non-finite image or segmentation')
    if not np.isin(label, [0, 1, 2, 3]).all():
        raise ValueError(f'{npz_path}: expected integer internal labels 0,1,2,3')
    if not np.any(label > 0):
        raise ValueError(f'{npz_path}: empty labeled training tumor mask')
    if affine.shape != (4, 4) or not np.isfinite(affine).all():
        raise ValueError(f'{npz_path}: invalid affine')
    if not np.allclose(affine[3], [0, 0, 0, 1]) or abs(np.linalg.det(affine[:3, :3])) <= 1e-8:
        raise ValueError(f'{npz_path}: affine must be a nonsingular voxel-to-world transform')
    if spacing.shape != (3,) or not np.isfinite(spacing).all() or np.any(spacing <= 0):
        raise ValueError(f'{npz_path}: spacing requires three positive finite values')
    if not np.allclose(spacing, nib.affines.voxel_sizes(affine), rtol=1e-4, atol=1e-4):
        raise ValueError(f'{npz_path}: spacing/affine mismatch')
    normalized_axes = affine[:3, :3] / spacing[np.newaxis, :]
    if not np.allclose(normalized_axes.T @ normalized_axes, np.eye(3), atol=1e-4):
        raise ValueError(f'{npz_path}: resample sheared geometry before surface metrics')
    return {'image': image.astype(np.float32, copy=False),
            'label': label.astype(np.uint8, copy=False), 'affine': affine, 'spacing': spacing}


def cache_is_current(npz_path: Path) -> bool:
    npz_path = Path(npz_path)
    if not npz_path.is_file() or npz_path.stat().st_size == 0:
        return False
    try:
        load_validated_cache(npz_path)
        return True
    except CACHE_ERRORS:
        return False


def preprocess_case(
    case_id: str,
    src_dir: Path,
    dst_dir: Path,
    force: bool = False,
) -> Path:
    """Validate, normalize and cache one BraTS case from raw NIfTI if not already cached."""
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    out_path = dst_dir / f"{case_id}.npz"
    if not force and cache_is_current(out_path):
        return out_path

    case_folder = Path(src_dir) / case_id
    if not case_folder.is_dir():
        raise FileNotFoundError(f"Case directory not found: {case_folder}")

    channels = []
    reference_shape = None
    reference_affine = None
    spacing = None
    for modality in MODALITIES:
        file_path = case_folder / f"{case_id}-{modality}.nii.gz"
        if not file_path.exists():
            raise FileNotFoundError(f"Missing modality for {case_id}: {file_path.name}")

        nii = nib.load(file_path)
        array = np.asarray(nii.dataobj, dtype=np.float32)
        if array.ndim != 3:
            raise ValueError(f"Expected a 3D volume for {file_path}, got shape {array.shape}")
        if not np.isfinite(array).all():
            raise ValueError(f"Non-finite MRI intensities in {file_path}")

        if reference_shape is None:
            reference_shape = array.shape
            reference_affine = np.asarray(nii.affine, dtype=np.float64)
            spacing = np.asarray(nib.affines.voxel_sizes(nii.affine), dtype=np.float32)
            if not np.isfinite(reference_affine).all() or abs(np.linalg.det(reference_affine[:3, :3])) <= 1e-8:
                raise ValueError(f"Invalid affine in {file_path}")
            if not np.isfinite(spacing).all() or np.any(spacing <= 0):
                raise ValueError(f"Invalid voxel spacing in {file_path}")
        else:
            if array.shape != reference_shape:
                raise ValueError(f"Modality shape mismatch in {case_id}: {array.shape} != {reference_shape}")
            if not np.allclose(nii.affine, reference_affine, rtol=0.0, atol=1e-4):
                raise ValueError(f"Modality affine mismatch in {case_id}: {file_path.name}")

        brain_mask = array > 0
        if not brain_mask.any():
            raise ValueError(f"Empty MRI signal in {file_path}")
        values = array[brain_mask]
        low, high = np.percentile(values, [0.5, 99.5])
        array = np.clip(array, low, high)
        mean_value = float(array[brain_mask].mean())
        std_value = float(array[brain_mask].std())
        if std_value <= 1e-6:
            raise ValueError(f"Near-zero intensity variance in {file_path}")
        array = (array - mean_value) / std_value
        array[~brain_mask] = 0.0
        channels.append(array.astype(np.float32, copy=False))

    image = np.stack(channels, axis=0)
    seg_path = case_folder / f"{case_id}-seg.nii.gz"
    if not seg_path.exists():
        raise FileNotFoundError(f"Missing training segmentation for {case_id}: {seg_path.name}")

    seg_nii = nib.load(seg_path)
    raw_seg = np.asarray(seg_nii.dataobj)
    if raw_seg.shape != reference_shape:
        raise ValueError(f"Segmentation shape mismatch in {case_id}: {raw_seg.shape} != {reference_shape}")
    if not np.isfinite(raw_seg).all():
        raise ValueError(f"Non-finite segmentation values in {case_id}")
    if not np.allclose(seg_nii.affine, reference_affine, rtol=0.0, atol=1e-4):
        raise ValueError(f"Segmentation affine mismatch in {case_id}")

    raw_labels = set(np.unique(raw_seg).tolist())
    unexpected = raw_labels - ALLOWED_RAW_LABELS
    if 3 in raw_labels and 4 in raw_labels:
        raise ValueError(f'Mixed ET encodings 3 and 4 in {case_id}; verify the dataset schema')
    if unexpected:
        raise ValueError(f"Unexpected labels in {case_id}: {sorted(unexpected)}")
    if not np.any(raw_seg > 0):
        raise ValueError(f"Empty tumor segmentation in {case_id}")

    label = np.zeros_like(raw_seg, dtype=np.uint8)
    for source_label, target_label in LABEL_MAP.items():
        label[raw_seg == source_label] = target_label
    label = label[None, ...]

    temp_path = out_path.with_suffix(".tmp")
    with temp_path.open("wb") as handle:
        np.savez_compressed(
            handle,
            image=image,
            label=label,
            seg=label[0],
            affine=reference_affine,
            spacing=spacing,
            preprocess_version=np.asarray(PREPROCESS_VERSION),
            case_id=np.asarray(case_id),
            modalities=np.asarray(MODALITIES),
        )
    temp_path.replace(out_path)
    return out_path


In [ ]:
def cache_content_inventory(cached: Dict[str, np.ndarray]) -> Dict:
    """Fingerprint decoded content during the validation read; no second read of Drive files."""
    digest = hashlib.sha256()
    for key in ('image', 'label', 'affine', 'spacing'):
        array = np.ascontiguousarray(cached[key])
        digest.update(json.dumps([key, array.dtype.str, list(array.shape)]).encode('utf-8'))
        digest.update(memoryview(array).cast('B'))
    counts = np.bincount(cached['label'].ravel(), minlength=4)
    return {'content_sha256': digest.hexdigest(), 'image_shape': list(cached['image'].shape),
            'spacing': cached['spacing'].tolist(), 'label_voxels': counts.tolist(),
            'et_voxels': int(counts[3])}



def write_json_atomic(value: Dict, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(destination)


def file_signature(path: Path) -> Dict:
    stat = path.stat()
    return {'size': stat.st_size, 'mtime_ns': stat.st_mtime_ns}


def persistent_cache_candidates(case_id: str) -> List[Path]:
    """Only known, explicitly configured directories; never scan all of Google Drive."""
    filename = f'{case_id}.npz'
    roots = [drive_npz_dir, *CACHE_FALLBACK_DIRS]
    return list(dict.fromkeys(Path(root) / filename for root in roots))


def resolve_persistent_cache(case_id: str, repair_missing: bool = True) -> Path:
    """Find an existing NPZ, or rebuild the missing original from raw NIfTI."""
    candidates = persistent_cache_candidates(case_id)
    for path in candidates:
        if path.is_file() and path.stat().st_size > 0:
            return path

    raw_case = Path(data_root) / case_id if data_root is not None else None
    if repair_missing and raw_case is not None and raw_case.is_dir():
        print(f'Rebuilding missing NPZ from raw MRI: {case_id}')
        return preprocess_case(case_id, Path(data_root), drive_npz_dir, force=True)

    alternatives = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(
        f'Missing persistent cache for {case_id}. Checked:\n{alternatives}\n'
        'If this folder is in Shared with me, add its shortcut to My Drive. '
        'If you only have another incompatible NPZ format, restore the raw NIfTI '
        'or the original matching NPZ cache; do not bypass validation.'
    )


def local_cache_path(case_id: str) -> Path:
    """Bounded on-disk copy; source may be a compatible alternate Drive directory."""
    source = resolve_persistent_cache(case_id)
    destination = local_npz_dir / source.name
    if source.resolve() == destination.resolve():
        return source
    local_npz_dir.mkdir(parents=True, exist_ok=True)
    source_signature = file_signature(source)
    # A local file is reusable only if it was copied from the current source.
    provenance_path = local_npz_dir / f'{case_id}.source.json'
    try:
        provenance = json.loads(provenance_path.read_text(encoding='utf-8'))
        if (destination.is_file() and file_signature(destination) == source_signature
                and provenance == {'path': str(source), 'signature': source_signature}):
            return destination
    except (FileNotFoundError, ValueError, OSError):
        pass
    limit = int(LOCAL_CACHE_GB * 1024 ** 3)
    if source_signature['size'] > limit:
        return source
    cached_files = []
    for candidate in local_npz_dir.glob('*.npz'):
        try:
            stat = candidate.stat()
            cached_files.append((stat.st_mtime_ns, stat.st_size, candidate))
        except FileNotFoundError:
            pass
    used = sum(item[1] for item in cached_files)
    for _, _, candidate in sorted(cached_files):
        if used + source_signature['size'] <= limit:
            break
        try:
            size = candidate.stat().st_size
            candidate.unlink()
            used -= size
            (local_npz_dir / f'{candidate.stem}.source.json').unlink(missing_ok=True)
        except FileNotFoundError:
            pass
        except PermissionError:
            pass
    if shutil.disk_usage(local_npz_dir).free < source_signature['size'] + 1024 ** 3:
        return source
    temporary = destination.with_suffix(f'.{os.getpid()}.{uuid.uuid4().hex}.tmp')
    try:
        shutil.copy2(source, temporary)
        temporary.replace(destination)
        metadata_tmp = provenance_path.with_suffix(f'.{os.getpid()}.tmp')
        try:
            metadata_tmp.write_text(json.dumps({'path': str(source), 'signature': source_signature}), encoding='utf-8')
            metadata_tmp.replace(provenance_path)
        finally:
            metadata_tmp.unlink(missing_ok=True)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


def prepare_dataset(drive_dir: Path, local_dir: Path, raw_dir: Optional[Path]) -> List[str]:
    """Validate cache content and retain the original experimental fingerprint."""
    global CACHE_INVENTORY, DATA_FINGERPRINT, CACHE_SOURCE_SIGNATURES
    drive_dir, local_dir = Path(drive_dir), Path(local_dir)
    drive_dir.mkdir(parents=True, exist_ok=True)
    local_dir.mkdir(parents=True, exist_ok=True)
    raw_dir = None if raw_dir is None else Path(raw_dir)

    raw_cases = (sorted(d.name for d in raw_dir.iterdir()
                        if d.is_dir() and is_brats_case_name(d.name))
                 if raw_dir is not None and raw_dir.is_dir() else [])
    cache_cases = sorted({p.stem for root in (drive_dir, *CACHE_FALLBACK_DIRS)
                          if Path(root).is_dir()
                          for p in Path(root).glob('*.npz')
                          if is_brats_case_name(p.stem)})
    cohort = raw_cases or cache_cases
    if not cohort:
        raise RuntimeError(
            f'No BraTS NPZ cases found. Primary: {drive_dir}; alternatives: {CACHE_FALLBACK_DIRS}. '
            'Check the Google Drive shortcut and rerun the configuration cell.'
        )
    if FORCE_REBUILD_CACHE and not raw_cases:
        raise RuntimeError('FORCE_REBUILD_CACHE requires raw NIfTI.')

    ledger_path = drive_dir / 'validated_inventory.json'
    try:
        ledger = json.loads(ledger_path.read_text(encoding='utf-8'))
    except (FileNotFoundError, ValueError):
        ledger = {}
    old = ledger.get('cases', {}) if ledger.get('preprocess_version') == PREPROCESS_VERSION else {}
    inventory, records, source_signatures = {}, {}, {}

    for index, case_id in enumerate(tqdm(cohort, desc='Prepare validated cohort')):
        npz_path = resolve_persistent_cache(case_id, repair_missing=bool(raw_cases))
        case_folder = raw_dir / case_id if raw_dir else None
        raw_paths = ([case_folder / f'{case_id}-{mod}.nii.gz' for mod in (*MODALITIES, 'seg')]
                     if case_folder else [])
        raw_signature = {p.name: file_signature(p) for p in raw_paths if p.is_file()}
        record = old.get(case_id, {})
        signature = file_signature(npz_path)
        # Legacy inventories had no path field; they always referred to the original cache.
        cached_record_path = record.get('cache_source', str(drive_dir / f'{case_id}.npz'))
        cache_unchanged = (cached_record_path == str(npz_path)
                           and record.get('npz_signature') == signature)
        raw_unchanged = record.get('raw_signature') == raw_signature
        if (not FORCE_REBUILD_CACHE and not REVALIDATE_CACHE
                and cache_unchanged and raw_unchanged and 'inventory' in record):
            item = record['inventory']
        else:
            rebuild = FORCE_REBUILD_CACHE or (bool(record) and not raw_unchanged and bool(raw_paths))
            if not rebuild:
                try:
                    cached = load_validated_cache(npz_path)
                except CACHE_ERRORS as exc:
                    if not case_folder:
                        raise ValueError(
                            f'{case_id}: cache incompatible or damaged at {npz_path}. '
                            f'Expected preprocessing={PREPROCESS_VERSION}; original error: {exc}. '
                            'The strict_v1 cache is not automatically equivalent to the v1 cache. '
                            'Restore the matching NPZ files or raw NIfTI.'
                        ) from exc
                    rebuild = True
            if rebuild:
                if not case_folder:
                    raise ValueError(f'{case_id}: invalid cache and no raw NIfTI available')
                npz_path = preprocess_case(case_id, raw_dir, drive_dir, force=True)
                cached = load_validated_cache(npz_path)
            item = cache_content_inventory(cached)
            del cached
        inventory[case_id] = item
        signature = file_signature(npz_path)
        source_signatures[case_id] = {'path': str(npz_path), 'signature': signature}
        records[case_id] = {
            'inventory': item, 'npz_signature': signature,
            'cache_source': str(npz_path), 'raw_signature': raw_signature,
        }
        if (index + 1) % 25 == 0:
            write_json_atomic({'preprocess_version': PREPROCESS_VERSION,
                               'cases': {**old, **records}}, ledger_path)
    write_json_atomic({'preprocess_version': PREPROCESS_VERSION, 'cases': records}, ledger_path)
    CACHE_INVENTORY = inventory
    CACHE_SOURCE_SIGNATURES = source_signatures
    DATA_FINGERPRINT = hashlib.sha256(json.dumps(
        {case: inventory[case]['content_sha256'] for case in sorted(inventory)},
        sort_keys=True).encode('utf-8')).hexdigest()
    print(f'Validated {len(cohort)} cases; fingerprint={DATA_FINGERPRINT}')
    print('Only cache files matching the expected contract can be used for this experiment.')
    return cohort


In [ ]:
all_cases = prepare_dataset(drive_npz_dir, local_npz_dir, data_root)
print(f"Valid cases: {len(all_cases)}")

sample_path = resolve_persistent_cache(all_cases[0], repair_missing=False)

start = time.time()
sample_data = load_validated_cache(sample_path)
read_duration = time.time() - start
sample_img = sample_data["image"]
sample_lbl = sample_data["label"]
sample_spacing = sample_data["spacing"]
sample_affine = sample_data["affine"]

print(f"Read time: {read_duration:.5f}s")
print(f"Image: {sample_img.shape} {sample_img.dtype}")
print(f"Label: {sample_lbl.shape} labels={np.unique(sample_lbl).tolist()}")
print(f"Spacing: {sample_spacing.tolist()} | affine determinant={np.linalg.det(sample_affine[:3, :3]):.4f}")

## 4. Phân chia tập dữ liệu và xây dựng DataLoaders

In [ ]:
def patient_id(case_id: str) -> str:
    """BraTS-GLI-XXXXX-VVV: keep all visits of a patient in one split."""
    match = re.fullmatch(r'(BraTS-[A-Za-z]+-\d+)-\d+', case_id)
    return match.group(1) if match else case_id


def validate_split_manifest(manifest: Dict, cases: List[str]) -> None:
    if manifest.get('data_fingerprint') != DATA_FINGERPRINT:
        raise ValueError('Dataset content changed or fingerprint is missing; choose a new experiment/manifest path')
    splits = [manifest[name] for name in ('train', 'val', 'test')]
    if any(not split for split in splits):
        raise ValueError('Train, validation and test must all be nonempty')
    flat = [case for split in splits for case in split]
    if len(flat) != len(set(flat)) or set(flat) != set(cases):
        raise ValueError('Manifest must contain each cohort case exactly once')
    patient_sets = [{patient_id(case) for case in split} for split in splits]
    if any(patient_sets[a] & patient_sets[b] for a, b in ((0, 1), (0, 2), (1, 2))):
        raise ValueError('A patient occurs in more than one split')
    if (manifest.get('seed') != SEED or manifest.get('val_fraction') != VAL_SPLIT
            or manifest.get('test_fraction') != TEST_SPLIT):
        raise ValueError('Manifest/config mismatch; choose a new manifest path for a new experiment')


if len(all_cases) < 20:
    raise RuntimeError(f'At least 20 labeled cases required, found {len(all_cases)}')
if not (0 < VAL_SPLIT < 1 and 0 < TEST_SPLIT < 1 and VAL_SPLIT + TEST_SPLIT < 1):
    raise ValueError('Invalid validation/test split fractions')
if SPLIT_MANIFEST_PATH.exists():
    split_manifest = json.loads(SPLIT_MANIFEST_PATH.read_text(encoding='utf-8'))
else:
    patients = sorted({patient_id(case) for case in all_cases})
    train_val_patients, test_patients = train_test_split(
        patients, test_size=TEST_SPLIT, random_state=SEED, shuffle=True)
    train_patients, val_patients = train_test_split(
        train_val_patients, test_size=VAL_SPLIT / (1 - TEST_SPLIT), random_state=SEED, shuffle=True)
    split_manifest = {
        'seed': SEED, 'val_fraction': VAL_SPLIT, 'test_fraction': TEST_SPLIT,
        'grouping': 'BraTS patient ID (strip final visit suffix)', 'cohort': sorted(all_cases),
        'data_fingerprint': DATA_FINGERPRINT,
        **{name: [case for case in sorted(all_cases) if patient_id(case) in set(group)]
           for name, group in [('train', train_patients), ('val', val_patients), ('test', test_patients)]},
    }
validate_split_manifest(split_manifest, all_cases)
split_manifest['cache_inventory'] = CACHE_INVENTORY
split_manifest['region_presence'] = {
    name: {'cases': len(split_manifest[name]),
           'et_positive_cases': sum(CACHE_INVENTORY[case]['et_voxels'] > 0 for case in split_manifest[name]),
           'et_absent_cases': sum(CACHE_INVENTORY[case]['et_voxels'] == 0 for case in split_manifest[name])}
    for name in ('train', 'val', 'test')}
if split_manifest['region_presence']['train']['et_positive_cases'] == 0:
    raise ValueError('No ET-positive training case. Check raw label schema/cache before training.')
for name, counts in split_manifest['region_presence'].items():
    print(f'{name} ET presence: {counts}')
    if counts['et_positive_cases'] == 0:
        print(f'WARNING: {name} has no ET-positive cases; ET Dice cannot measure tumor detection here.')
SPLIT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
write_json_atomic(split_manifest, SPLIT_MANIFEST_PATH)
train_cases, val_cases, test_cases = (split_manifest[name] for name in ('train', 'val', 'test'))
rng = np.random.default_rng(SEED)
val_cases_fast = sorted(rng.choice(val_cases, size=min(VAL_SUBSET_SIZE, len(val_cases)), replace=False).tolist())
print(f'Train: {len(train_cases)}')
print(f'Validation: {len(val_cases)} (fast monitor: {len(val_cases_fast)})')
print(f'Held-out test: {len(test_cases)}')
print(f'Split manifest: {SPLIT_MANIFEST_PATH}')


In [ ]:
class BraTSDataset3D(Dataset):
    def __init__(self, npz_dir: Union[str, Path], case_list: List[str], transforms=None):
        self.npz_dir = Path(npz_dir)
        self.case_list = list(case_list)
        self.transforms = transforms
        self.epoch = 1
        self.deterministic_aug = False

    def __len__(self) -> int:
        return len(self.case_list)

    def __getitem__(self, index: int) -> Dict:
        case_id = self.case_list[index]
        npz_path = local_cache_path(case_id)
        try:
            cached = load_validated_cache(npz_path)
        except FileNotFoundError:
            # A second worker may have evicted the local disk copy.
            fallback = resolve_persistent_cache(case_id, repair_missing=False)
            cached = load_validated_cache(fallback)
        except Exception as exc:
            raise RuntimeError(
                f'Invalid or unreadable cache for {case_id}: {npz_path}. '
                'Check the cache version and rerun data preparation.'
            ) from exc

        sample = {
            "image": cached["image"],
            "label": cached["label"].astype(np.int64),
            "case_id": case_id,
            "affine": cached["affine"],
            "spacing": cached["spacing"],
        }
        if self.transforms is None:
            return sample
        if self.deterministic_aug:
            # Per-case deterministic augmentation independent of resumed batch offset.
            # Safe because training uses num_workers=0.
            rng_seed = int((SEED * 1000003 + self.epoch * 1009 + index) % (2 ** 32 - 1))
            self.transforms.set_random_state(seed=rng_seed)
        return self.transforms(sample)


In [ ]:
train_transforms = mt.Compose([
    mt.SpatialPadd(keys=['image', 'label'], spatial_size=PATCH_SIZE, mode='constant'),
    mt.RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=PATCH_SIZE,
        pos=3.0,
        neg=1.0,
        num_samples=NUM_SAMPLES,
        allow_smaller=False,
    ),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    mt.RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    # Co giãn nhẹ mô phỏng sai lệch không gian giữa các bệnh nhân
    mt.RandAffined(
        keys=["image", "label"],
        prob=0.2,
        scale_range=(0.05, 0.05, 0.05),
        mode=["bilinear", "nearest"],
        padding_mode="zeros",
    ),
    #  Nhiễu Gaussian mô phỏng nhiễu hạt MRI từ máy chụp thực tế
    mt.RandGaussianNoised(keys="image", prob=0.2, std=0.05),
    mt.RandScaleIntensityd(keys="image", factors=0.1, prob=0.5),
    mt.RandShiftIntensityd(keys="image", offsets=0.1, prob=0.5),
    mt.EnsureTyped(keys=["image", "label"], track_meta=False),
])


eval_transforms = mt.Compose([
    mt.EnsureTyped(keys=["image", "label"], track_meta=False),
])


In [ ]:
# The V1_TRAIN preflight inspects every case_id before starting worker processes.
train_dataset = BraTSDataset3D(local_npz_dir, train_cases, transforms=train_transforms)
train_dataset.deterministic_aug = True
val_dataset_fast = BraTSDataset3D(local_npz_dir, val_cases_fast, transforms=eval_transforms)
val_dataset_full = BraTSDataset3D(local_npz_dir, val_cases, transforms=eval_transforms)
test_dataset = BraTSDataset3D(local_npz_dir, test_cases, transforms=eval_transforms)

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
}
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 1
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = MonaiDataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Smoke-test only. Actual training builds epoch-permutation batches.
    persistent_workers=False,
    generator=loader_generator,
    **loader_kwargs,
)
val_loader_fast = MonaiDataLoader(
    val_dataset_fast, batch_size=1, shuffle=False, persistent_workers=False, **loader_kwargs
)
val_loader_full = MonaiDataLoader(
    val_dataset_full, batch_size=1, shuffle=False, persistent_workers=False, **loader_kwargs
)
test_loader = MonaiDataLoader(
    test_dataset, batch_size=1, shuffle=False, persistent_workers=False, **loader_kwargs
)

epoch_steps = len(train_loader) if STEPS_PER_EPOCH is None else min(len(train_loader), STEPS_PER_EPOCH)
print(f"Train batches/epoch: {epoch_steps}/{len(train_loader)}; {NUM_SAMPLES} patches/case")
print(f"Fast validation cases: {len(val_dataset_fast)}")
print(f"Full validation cases: {len(val_dataset_full)}")
print(f"Held-out test cases: {len(test_dataset)}")


## 5. Metrics va evaluation

Dice, IoU, Precision, Recall, HD95 va the tich duoc tinh rieng cho WT, TC va ET.
HD95 la dinh nghia noi bo: percentile 95 cua khoang cach hai huong gop lai,
mat bien 26-neighbor; hai mask rong cho Dice/IoU = 1, HD95 = 0;
mot mask rong cho Dice = 0, HD95 = duong cheo volume. Neu ro quy uoc khi bao cao.
Dice duoc macro-average theo ca va WT/TC/ET.
Metric test chi duoc dung de bao cao cuoi, khong dung de chon model.


In [ ]:
REGION_BUILDERS = {
    "wt": lambda array: array > 0,
    "tc": lambda array: np.isin(array, [1, 3]),
    "et": lambda array: array == 3,
}


def _binary_metrics(
    prediction: np.ndarray,
    target: np.ndarray,
    spacing: Tuple[float, float, float],
    include_surface: bool = True,
) -> Dict[str, float]:
    prediction = prediction.astype(bool)
    target = target.astype(bool)
    pred_count = int(prediction.sum())
    target_count = int(target.sum())
    intersection = int(np.logical_and(prediction, target).sum())
    union = pred_count + target_count - intersection

    if pred_count == 0 and target_count == 0:
        dice = iou = precision = recall = 1.0
        hd95 = 0.0
    else:
        dice = 2.0 * intersection / max(pred_count + target_count, 1)
        iou = intersection / max(union, 1)
        precision = intersection / pred_count if pred_count else 0.0
        recall = intersection / target_count if target_count else 0.0
        hd95 = float("nan")
        if include_surface:
            if pred_count == 0 or target_count == 0:
                hd95 = float(np.linalg.norm(np.asarray(prediction.shape) * np.asarray(spacing)))
            else:
                structure = np.ones((3, 3, 3), dtype=bool)
                pred_surface = np.logical_xor(
                    prediction,
                    binary_erosion(prediction, structure=structure, border_value=0),
                )
                target_surface = np.logical_xor(
                    target,
                    binary_erosion(target, structure=structure, border_value=0),
                )
                distance_to_target = distance_transform_edt(~target_surface, sampling=spacing)
                distance_to_pred = distance_transform_edt(~pred_surface, sampling=spacing)
                distances = np.concatenate([
                    distance_to_target[pred_surface],
                    distance_to_pred[target_surface],
                ])
                hd95 = float(np.percentile(distances, 95))

    voxel_volume_cm3 = float(np.prod(spacing) / 1000.0)
    result = {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "volume_cm3": float(pred_count * voxel_volume_cm3),
    }
    if include_surface:
        result["hd95"] = float(hd95)
    return result


def compute_case_metrics(
    prediction: Union[np.ndarray, torch.Tensor],
    target: Union[np.ndarray, torch.Tensor],
    spacing: Tuple[float, float, float],
    include_surface: bool = True,
) -> Dict[str, float]:
    if isinstance(prediction, torch.Tensor):
        prediction = prediction.detach().cpu().numpy()
    if isinstance(target, torch.Tensor):
        target = target.detach().cpu().numpy()
    prediction = np.asarray(prediction)
    target = np.asarray(target)
    while prediction.ndim > 3 and prediction.shape[0] == 1:
        prediction = prediction[0]
    while target.ndim > 3 and target.shape[0] == 1:
        target = target[0]
    if prediction.ndim != 3 or target.ndim != 3:
        raise ValueError('Metrics require one 3D label volume per case')
    spacing = np.asarray(spacing, dtype=np.float64)
    if spacing.shape != (3,) or not np.isfinite(spacing).all() or np.any(spacing <= 0):
        raise ValueError('Metrics require three positive finite voxel spacings')
    if not np.isin(prediction, [0, 1, 2, 3]).all() or not np.isin(target, [0, 1, 2, 3]).all():
        raise ValueError('Metrics expect internal labels 0,1,2,3')
    if prediction.shape != target.shape:
        raise ValueError(f"Metric shape mismatch: {prediction.shape} != {target.shape}")

    result = {}
    for region_name, builder in REGION_BUILDERS.items():
        region_metrics = _binary_metrics(
            builder(prediction),
            builder(target),
            spacing,
            include_surface=include_surface,
        )
        for metric_name, value in region_metrics.items():
            result[f"{metric_name}_{region_name}"] = value
    result["dice_mean"] = float(np.mean([result[f"dice_{name}"] for name in REGION_BUILDERS]))
    result["iou_mean"] = float(np.mean([result[f"iou_{name}"] for name in REGION_BUILDERS]))
    result["precision_mean"] = float(np.mean([result[f"precision_{name}"] for name in REGION_BUILDERS]))
    result["recall_mean"] = float(np.mean([result[f"recall_{name}"] for name in REGION_BUILDERS]))
    if include_surface:
        result["hd95_mean"] = float(np.mean([result[f"hd95_{name}"] for name in REGION_BUILDERS]))
    return result

def postprocess_remove_small_objects(mask_np: np.ndarray, min_size: int = 50) -> np.ndarray:
    """Loại bỏ các đốm u nhiễu nhỏ hơn min_size voxel"""
    cleaned = mask_np.copy()
    for c in [1, 2, 3]:
        labeled, n_features = nd_label(cleaned == c)
        counts = np.bincount(labeled.ravel())
        small_components = counts < min_size
        small_components[0] = False
        cleaned[small_components[labeled]] = 0
    return cleaned

@torch.inference_mode()
def predict_volume(model: torch.nn.Module, images: torch.Tensor,
                   use_tta: Optional[bool] = None) -> torch.Tensor:
    """Only 96³ patches go to GPU; stitch float32 probabilities on CPU."""
    model.eval()
    if images.ndim != 5 or images.shape[1] != 4:
        raise ValueError('Inference expects (B,4,H,W,D)')
    use_tta = INFERENCE_USE_TTA if use_tta is None else use_tta
    images = images.detach().to(device='cpu', dtype=torch.float32)
    model_device = next(model.parameters()).device

    def infer(volume):
        # Accumulate in float32; autocast applies only to the model's GPU patches.
        def predictor(patches):
            with torch.autocast(device_type=model_device.type, enabled=model_device.type == 'cuda'):
                return model(patches).float()
        logits = sliding_window_inference(
            inputs=volume, roi_size=PATCH_SIZE, sw_batch_size=SW_BATCH_SIZE,
            predictor=predictor, overlap=SW_OVERLAP, mode='gaussian',
            sw_device=model_device, device=torch.device('cpu'))
        return torch.softmax(logits, dim=1)

    probabilities = infer(images)
    if use_tta:
        flipped = infer(torch.flip(images, dims=[2]))
        probabilities.add_(torch.flip(flipped, dims=[2])).mul_(0.5)
        del flipped
    masks = probabilities.argmax(dim=1).numpy().astype(np.uint8)
    cleaned = np.stack([postprocess_remove_small_objects(mask, SMALL_COMPONENT_VOXELS) for mask in masks])
    return torch.from_numpy(cleaned[:, None])


@torch.no_grad()
def evaluate_model(
    model: torch.nn.Module,
    loader: MonaiDataLoader,
    description: str,
    include_surface: bool = True,
) -> Tuple[List[Dict], Dict[str, float]]:
    model.eval()
    case_results = []
    for batch in tqdm(loader, desc=description, leave=False):
        if batch['image'].shape[0] != 1:
            raise ValueError('Evaluation loader must use batch_size=1 for per-case metadata')
        started = time.perf_counter()
        predictions = predict_volume(model, batch['image'])
        inference_seconds = time.perf_counter() - started
        spacing = tuple(float(value) for value in batch["spacing"][0].cpu().numpy())
        metrics = compute_case_metrics(
            predictions[0],
            batch["label"][0],
            spacing,
            include_surface=include_surface,
        )
        metrics['inference_seconds'] = inference_seconds
        metrics['target_et_present'] = bool(torch.any(batch['label'] == 3).item())
        metrics['case_id'] = batch['case_id'][0]
        case_results.append(metrics)

    if not case_results:
        raise RuntimeError(f"No cases were evaluated for {description}")
    numeric_keys = [key for key in case_results[0] if key not in ('case_id', 'target_et_present')]
    summary = {
        key: float(np.nanmean([case[key] for case in case_results]))
        for key in numeric_keys
    }
    summary['case_count'] = len(case_results)
    summary['et_positive_cases'] = sum(case['target_et_present'] for case in case_results)
    summary['et_absent_cases'] = len(case_results) - summary['et_positive_cases']
    positive = [case['dice_et'] for case in case_results if case['target_et_present']]
    summary['dice_et_positive'] = float(np.mean(positive)) if positive else None
    if summary['et_positive_cases'] == 0:
        print(f'WARNING: {description} has no ET-positive targets; do not interpret ET Dice as detection performance.')
    return case_results, summary


## 6. Model factory va loss

Hai kien truc dung chung training/evaluation contract de ket qua so sanh nhat quan.

In [ ]:
def build_model(model_name: str = 'unet3d') -> torch.nn.Module:
    if model_name != 'unet3d':
        raise ValueError('v1 trains exactly one 3D U-Net; no second model is needed.')
    return UNet(
        spatial_dims=3, in_channels=4, out_channels=4,
        channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
        num_res_units=2, norm='instance', dropout=0.2,
    )

probe_model = build_model()
print(f'3D U-Net: {sum(p.numel() for p in probe_model.parameters()):,} parameters')
del probe_model


In [ ]:
criterion = DiceCELoss(
    to_onehot_y=True,
    softmax=True,
    include_background=False,
    lambda_dice=1.0,
    lambda_ce=1.0,
)

MODEL_RUNS: Dict[str, Dict] = {}

## 7. Train 3D U-Net — checkpoint từng 25 mini-batch

- **Bấm chạy cell bên dưới**: tự train từ bước chưa hoàn thành. Mỗi 25 batch lưu `unet3d_brats_in_progress.pth` lên Drive; cuối epoch lưu `unet3d_brats_last.pth`, sau full-validation mới lưu `unet3d_brats_best.pth`.
- Nếu Colab ngắt, **chạy lại notebook từ đầu**. Model/optimizer/scheduler/scaler được phục hồi từ checkpoint phù hợp nhất và những batch đã hoàn thành không được train lặp.
- Mặc định `MAX_EPOCHS=50`, `EPOCHS_PER_SESSION=50`, `SAVE_EVERY_STEPS=25`. Nếu phiên Colab bị giới hạn, lần chạy sau tiếp tục. Tham số `STEPS_PER_EPOCH=None` giữ nguyên số batch của tập train; không giảm tập train để tạo kết quả giả.
- **Mục 8–10 sẽ bỏ qua an toàn** khi train chưa xong. Sau hoàn tất, chạy lại các mục này để xuất biểu đồ/báo cáo.

In [ ]:
# CELL: V2_STATUS — look up saved training progress without requiring a checkpoint
progress_file = reports_dir / 'training_progress.json'
if progress_file.is_file():
    progress = json.loads(progress_file.read_text(encoding='utf-8'))
    print('Training progress:', progress.get('epoch', 0), '/', MAX_EPOCHS, 'finished epochs')
    if progress.get('partial_epoch'):
        print('Partially finished epoch:', progress['partial_epoch'],
              'batch:', progress.get('step_in_epoch'), '/', progress.get('steps_this_epoch'))
    print('Best validation Dice:', progress.get('best_dice'))
    print('Completed:', progress.get('complete', False))
else:
    print('No checkpoint/progress in this new V2 experiment — starts at epoch 1.')
print('Checkpoint directory:', checkpoints_dir)


In [ ]:
# CELL: V2_SMOKE_TEST - checks real train data and augmentation
from itertools import islice
try:
    b = next(iter(train_loader))
except Exception as exc:
    raise RuntimeError('DataLoader/augmentation failed before training; check NPZ and transforms.') from exc
x, y = b['image'], b['label']
if x.ndim != 5 or x.shape[1] != 4 or y.ndim != 5 or y.shape[1] != 1:
    raise ValueError(f'Unexpected training batch: image={tuple(x.shape)}, label={tuple(y.shape)}')
if not torch.isfinite(x).all() or not torch.isfinite(y).all():
    raise ValueError('NaN/Inf found in first training batch')
if not torch.all((y >= 0) & (y <= 3)):
    raise ValueError('Labels must be 0,1,2,3 after mapping')
print('BATCH TEST PASS:', 'image', tuple(x.shape), 'label', tuple(y.shape))
del b, x, y


In [ ]:
# CELL: V2_TRAIN - resume an unfinished epoch from saved mini-batch
from torch.optim.lr_scheduler import CosineAnnealingLR

LAST_CKPT = checkpoints_dir / 'unet3d_brats_last.pth'
PARTIAL_CKPT = checkpoints_dir / 'unet3d_brats_in_progress.pth'
BEST_CKPT = checkpoints_dir / 'unet3d_brats_best.pth'
PROGRESS_JSON = reports_dir / 'training_progress.json'


class ProtocolMismatchError(ValueError):
    pass


def train_protocol():
    # This entire contract is checked before restoring weights AND optimizer.
    return {
        'version': 4,
        'experiment_id': EXPERIMENT_ID,
        'model': 'unet3d',
        'architecture': {'channels': [16, 32, 64, 128, 256], 'strides': [2, 2, 2, 2],
                         'num_res_units': 2, 'norm': 'instance', 'dropout': 0.2},
        'data_fingerprint': DATA_FINGERPRINT,
        'preprocess_version': PREPROCESS_VERSION,
        'labels': RAW_LABEL_SCHEMA,
        'modalities': list(MODALITIES),
        'split': {name: list(split_manifest[name]) for name in ('train', 'val', 'test')},
        'seed': SEED, 'patch_size': list(PATCH_SIZE),
        'batch_size': BATCH_SIZE, 'num_samples': NUM_SAMPLES,
        'total_epochs': MAX_EPOCHS, 'steps_per_epoch': STEPS_PER_EPOCH,
        'augmentation': 'deterministic_per_case_per_epoch_v2',
        'lr': LR, 'weight_decay': WEIGHT_DECAY,
        'fast_val_interval': FAST_VAL_INTERVAL,
        'full_val_interval': FULL_VAL_INTERVAL,
        'fast_val_cases': list(val_cases_fast),
        'early_stopping': [EARLY_STOP_PATIENCE, EARLY_STOP_MIN_DELTA],
        'inference': [INFERENCE_USE_TTA, SMALL_COMPONENT_VOXELS, SW_BATCH_SIZE,
                      SW_OVERLAP, METRIC_POLICY],
        'torch_version': str(torch.__version__),
        'monai_version': __import__('monai').__version__,
    }


def checkpoint_atomic(payload, target):
    """Never modify an existing checkpoint before the new file is complete."""
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_name(target.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        torch.save(payload, tmp)
        # Preserve a recovery backup of last/partial before overwriting.
        if target.is_file() and target in (LAST_CKPT, PARTIAL_CKPT):
            old_backup = target.with_suffix('.prev.pth')
            backup_tmp = old_backup.with_name(old_backup.name + '.tmp')
            try:
                shutil.copy2(target, backup_tmp)
                backup_tmp.replace(old_backup)
            finally:
                backup_tmp.unlink(missing_ok=True)
        tmp.replace(target)
    finally:
        tmp.unlink(missing_ok=True)


def load_checked_checkpoint(path, expected_protocol):
    ckpt = torch.load(path, map_location='cpu', weights_only=True)
    if ckpt.get('protocol') != expected_protocol:
        raise ProtocolMismatchError(
            f'{path.name}: khac cau hinh, tap du lieu hoac phien ban. '
            'Khong duoc ep resume. Hay dung lai config cu hoac tao EXPERIMENT_ID moi.'
        )
    return ckpt


def load_best_checkpoint(path, expected_protocol):
    return load_checked_checkpoint(path, expected_protocol)


def find_resume_state(protocol):
    files = [PARTIAL_CKPT, PARTIAL_CKPT.with_suffix('.prev.pth'),
             LAST_CKPT, LAST_CKPT.with_suffix('.prev.pth'), BEST_CKPT]
    candidates = []
    errors = []
    for path in files:
        if not path.is_file():
            continue
        try:
            state = load_checked_checkpoint(path, protocol)
            epoch = int(state['epoch'])
            step = int(state.get('step_in_epoch', 0))
            if step < 0 or epoch < 0 or (step and epoch >= MAX_EPOCHS):
                raise ValueError('Invalid epoch/step stored in checkpoint')
            candidates.append((epoch, step, path, state))
        except ProtocolMismatchError:
            raise
        except Exception as ex:
            errors.append(f'{path}: {ex}')
    if not candidates:
        if errors:
            raise RuntimeError('Checkpoint corrupt or unreadable:\n' + '\n'.join(errors))
        return None
    _, _, path, state = max(candidates, key=lambda row: (row[0], row[1]))
    best_epoch = int(state.get('best_epoch', -1))
    if best_epoch > 0:
        if not BEST_CKPT.is_file():
            raise FileNotFoundError('Best checkpoint missing. Restore unet3d_brats_best.pth.')
        best = load_checked_checkpoint(BEST_CKPT, protocol)
        if (int(best['epoch']) != best_epoch
                or best.get('state_id') != state.get('best_state_id')):
            raise ValueError('Best and last checkpoints are not a consistent pair.')
    print('Resume:', path, '| completed epoch:', state['epoch'],
          '| saved step in next epoch:', state.get('step_in_epoch', 0))
    return state


def deterministic_batches(epoch, start_step=0):
    """Resuming does not replay prior training samples or optimizer steps."""
    rng = np.random.default_rng(SEED + epoch)
    ids = rng.permutation(len(train_dataset)).tolist()
    batches = [ids[i:i + BATCH_SIZE] for i in range(0, len(ids), BATCH_SIZE)]
    if STEPS_PER_EPOCH is not None:
        batches = batches[:STEPS_PER_EPOCH]
    if not batches:
        raise RuntimeError('No train batches were scheduled.')
    total = len(batches)
    if start_step > total:
        raise ValueError('Saved partial checkpoint step > current epoch batch count.')
    train_dataset.epoch = epoch
    loader = MonaiDataLoader(
        train_dataset,
        batch_sampler=batches[start_step:],
        num_workers=0,  # Essential for reproducible per-case transforms and safe cache eviction
        pin_memory=(device.type == 'cuda'),
    )
    return loader, total


def blank_history():
    return {'train_loss': [], 'fast_val_epochs': [], 'fast_val_dice': [],
            'full_val_epochs': [], 'full_val_dice': [], 'learning_rates': [],
            'train_seconds': []}


def pack_training_state(model, optimizer, scheduler, scaler, *, epoch, step,
                        loss_sum, elapsed_seconds, history, best_epoch, best_score,
                        best_state_id, stale_checks, stopping_reference, complete):
    return {
        'protocol': train_protocol(), 'state_id': uuid.uuid4().hex,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'epoch': int(epoch),  # number of FULLY completed epochs
        'step_in_epoch': int(step),  # batches completed in next epoch
        'epoch_loss_sum': float(loss_sum),
        'epoch_elapsed_seconds': float(elapsed_seconds),
        'history': history,
        'best_epoch': int(best_epoch), 'best_score': float(best_score),
        'best_state_id': best_state_id,
        'stale_checks': int(stale_checks),
        'stopping_reference': float(stopping_reference),
        'complete': bool(complete),
    }


def training_preflight():
    # DataLoaders must exist; check their real case_list rather than .data.
    names = ('train_loader', 'val_loader_fast', 'val_loader_full', 'test_loader')
    all_ids = set()
    for name in names:
        loader = globals().get(name)
        if loader is None or not hasattr(loader.dataset, 'case_list'):
            raise RuntimeError(f'{name} chua duoc tao. Chay lai muc 4 truoc.')
        all_ids.update(loader.dataset.case_list)
    missing = []
    for case_id in sorted(all_ids):
        try:
            record = CACHE_SOURCE_SIGNATURES[case_id]
            src = Path(record['path'])
            if not src.is_file() or file_signature(src) != record['signature']:
                # Source may have moved to another configured directory; compare content.
                candidate = resolve_persistent_cache(case_id, repair_missing=True)
                sha = cache_content_inventory(load_validated_cache(candidate))['content_sha256']
                if sha != CACHE_INVENTORY[case_id]['content_sha256']:
                    raise ValueError('Content differs from verified cohort fingerprint')
                CACHE_SOURCE_SIGNATURES[case_id] = {
                    'path': str(candidate), 'signature': file_signature(candidate)
                }
        except Exception as ex:
            missing.append(f'{case_id}: {ex}')
    if missing:
        raise FileNotFoundError('Missing/incompatible cache files:\n' + '\n'.join(missing[:20]))
    print(f'Preflight PASS: {len(all_ids)} unique NPZ records.')


def run_training_session():
    training_preflight()
    protocol = train_protocol()
    restored = find_resume_state(protocol) if RESUME_V1 else None
    if not RESUME_V1 and any(p.exists() for p in (LAST_CKPT, PARTIAL_CKPT, BEST_CKPT)):
        raise FileExistsError('There are existing checkpoints. Enable RESUME_V1 or use another EXPERIMENT_ID.')

    model = build_model('unet3d').to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
    history = blank_history()
    last_epoch, start_step = 0, 0
    loss_sum, partial_seconds = 0., 0.
    best_epoch, best_score, best_state_id = -1, -1., None
    stale_checks, stopping_reference = 0, -1.
    if restored is not None:
        model.load_state_dict(restored['model_state'])
        optimizer.load_state_dict(restored['optimizer_state'])
        scheduler.load_state_dict(restored['scheduler_state'])
        scaler.load_state_dict(restored['scaler_state'])
        last_epoch = int(restored['epoch'])
        start_step = int(restored.get('step_in_epoch', 0))
        loss_sum = float(restored.get('epoch_loss_sum', 0.))
        partial_seconds = float(restored.get('epoch_elapsed_seconds', 0.))
        history = restored['history']
        best_epoch = int(restored['best_epoch'])
        best_score = float(restored['best_score'])
        best_state_id = restored['best_state_id']
        stale_checks = int(restored['stale_checks'])
        stopping_reference = float(restored['stopping_reference'])
        del restored

    if last_epoch >= MAX_EPOCHS or stale_checks >= EARLY_STOP_PATIENCE:
        complete = True
        print(f'ALREADY COMPLETE. Best epoch {best_epoch} (Dice={best_score:.4f}).')
    else:
        complete = False
        final_epoch = min(MAX_EPOCHS, last_epoch + EPOCHS_PER_SESSION)
        print(f'Training {last_epoch + 1}..{final_epoch} / {MAX_EPOCHS}. '
              f'Autosave every {SAVE_EVERY_STEPS} batches.')
        for epoch in range(last_epoch + 1, final_epoch + 1):
            pending = start_step if epoch == last_epoch + 1 else 0
            epoch_loss = loss_sum if pending else 0.
            spent = partial_seconds if pending else 0.
            loader, total_steps = deterministic_batches(epoch, start_step=pending)
            model.train()
            started = time.perf_counter()
            for step, batch in enumerate(tqdm(loader, total=total_steps-pending,
                                               desc=f'3D U-Net epoch {epoch}/{MAX_EPOCHS}'),
                                         start=pending + 1):
                x = batch['image'].to(device, non_blocking=True)
                y = batch['label'].to(device, non_blocking=True).long()
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                    logits = model(x)
                    loss = criterion(logits, y)
                if not bool(torch.isfinite(loss).all().item()):
                    raise FloatingPointError(f'Non-finite loss at epoch {epoch}, step {step}')
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                epoch_loss += float(loss.item())
                del x, y, logits, loss

                # Save COMPLETE optimizer state mid-epoch, not just model weights.
                if step % SAVE_EVERY_STEPS == 0 or step == total_steps:
                    partial = pack_training_state(
                        model, optimizer, scheduler, scaler,
                        epoch=epoch-1, step=step, loss_sum=epoch_loss,
                        elapsed_seconds=spent + time.perf_counter() - started,
                        history=history, best_epoch=best_epoch, best_score=best_score,
                        best_state_id=best_state_id, stale_checks=stale_checks,
                        stopping_reference=stopping_reference, complete=False
                    )
                    checkpoint_atomic(partial, PARTIAL_CKPT)
                    write_json_atomic({'epoch': epoch-1, 'partial_epoch': epoch,
                                       'step_in_epoch': step, 'steps_this_epoch': total_steps,
                                       'best_epoch': best_epoch, 'best_dice': best_score,
                                       'complete': False}, PROGRESS_JSON)
                    del partial

            duration = spent + time.perf_counter() - started
            current_lr = float(optimizer.param_groups[0]['lr'])
            scheduler.step()
            history['train_loss'].append(epoch_loss / total_steps)
            history['learning_rates'].append(current_lr)
            history['train_seconds'].append(duration)
            start_step, loss_sum, partial_seconds = 0, 0., 0.
            print(f'Epoch {epoch}/{MAX_EPOCHS}: train_loss={history["train_loss"][-1]:.4f}')

            # Full validation is expensive: first complete full pass at epoch 5.
            full_due = epoch % FULL_VAL_INTERVAL == 0 or epoch == MAX_EPOCHS
            if full_due:
                _, full = evaluate_model(model, val_loader_full,
                                         f'full validation ep{epoch}', include_surface=False)
                score = float(full['dice_mean'])
                if not np.isfinite(score):
                    raise FloatingPointError('Full validation produced non-finite Dice.')
                history['full_val_epochs'].append(epoch)
                history['full_val_dice'].append(score)
                is_best = score > best_score
                if is_best:
                    best_epoch, best_score = epoch, score
                if score > stopping_reference + EARLY_STOP_MIN_DELTA:
                    stopping_reference, stale_checks = score, 0
                else:
                    stale_checks += 1
                print(f'Full validation Dice={score:.4f}; best epoch={best_epoch}')
            else:
                is_best = False
                if epoch % FAST_VAL_INTERVAL == 0:
                    _, fast = evaluate_model(model, val_loader_fast,
                                             f'fast validation ep{epoch}', include_surface=False)
                    history['fast_val_epochs'].append(epoch)
                    history['fast_val_dice'].append(float(fast['dice_mean']))
                    print(f'Fast validation Dice={float(fast["dice_mean"]):.4f}')

            complete = epoch >= MAX_EPOCHS or stale_checks >= EARLY_STOP_PATIENCE
            state = pack_training_state(
                model, optimizer, scheduler, scaler,
                epoch=epoch, step=0, loss_sum=0., elapsed_seconds=0.,
                history=history, best_epoch=best_epoch, best_score=best_score,
                best_state_id=best_state_id, stale_checks=stale_checks,
                stopping_reference=stopping_reference, complete=complete
            )
            if is_best:
                best_state_id = state['state_id']
                state['best_state_id'] = best_state_id
                checkpoint_atomic(state, BEST_CKPT)
            checkpoint_atomic(state, LAST_CKPT)
            write_json_atomic({'epoch': epoch, 'best_epoch': best_epoch,
                               'best_dice': best_score, 'complete': complete,
                               'history': history}, PROGRESS_JSON)
            PARTIAL_CKPT.unlink(missing_ok=True)
            PARTIAL_CKPT.with_suffix('.prev.pth').unlink(missing_ok=True)
            last_epoch = epoch
            print(f'Epoch {epoch} checkpoint saved. Complete={complete}')
            if complete:
                break

    result = {
        'model_name': 'unet3d', 'checkpoint_path': BEST_CKPT,
        'history': history, 'best_epoch': best_epoch, 'protocol': protocol,
        'complete': complete, 'last_epoch': last_epoch,
    }
    del model, optimizer, scheduler, scaler
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return result


MODEL_RUNS = {}
RUN_COMPLETE = False
best_model = None
best_checkpoint = None
best_model_name = 'unet3d'
best_run = run_training_session()
MODEL_RUNS[best_model_name] = best_run
TRAINING_COMPLETE = bool(best_run['complete'])
# Final reports are unlocked only after final validation/test finish.
RUN_COMPLETE = False

if not TRAINING_COMPLETE:
    print(f'Current status: {best_run["last_epoch"]}/{MAX_EPOCHS} completed epochs.')
    print('Run V2_TRAIN again to continue. The most recent mini-batch checkpoint is on Drive.')
else:
    best_checkpoint = load_best_checkpoint(BEST_CKPT, best_run['protocol'])
    best_model = build_model().to(device)
    best_model.load_state_dict(best_checkpoint['model_state'])
    best_model.eval()
    evaluation_path = reports_dir / 'unet3d_evaluation.json'
    evaluation = json.loads(evaluation_path.read_text(encoding='utf-8')) if evaluation_path.is_file() else None
    if evaluation is None or (evaluation.get('protocol') != best_run['protocol'] or
                              evaluation.get('best_state_id') != best_checkpoint['state_id']):
        print('Evaluating FINAL selected model (validation + held-out test, once)...')
        val_cases_out, val_summary = evaluate_model(best_model, val_loader_full,
                                                    'final validation', include_surface=True)
        test_cases_out, test_summary = evaluate_model(best_model, test_loader,
                                                      'held-out test', include_surface=True)
        evaluation = {
            'protocol': best_run['protocol'], 'best_state_id': best_checkpoint['state_id'],
            'val_cases': val_cases_out, 'val_summary': val_summary,
            'test_cases': test_cases_out, 'test_summary': test_summary,
        }
        write_json_atomic(evaluation, evaluation_path)
    else:
        print('Reused matching final evaluation from Drive.')
    for key in ('val_cases', 'val_summary', 'test_cases', 'test_summary'):
        best_run[key] = evaluation[key]
    for split in ('val', 'test'):
        items = evaluation[f'{split}_cases']
        if items:
            with (reports_dir / f'{split}_per_case.csv').open('w', newline='', encoding='utf-8') as fh:
                writer = csv.DictWriter(fh, fieldnames=list(items[0]))
                writer.writeheader()
                writer.writerows(items)
    RUN_COMPLETE = True
    print('FINAL TRAINING AND EVALUATION COMPLETE. Best checkpoint:', BEST_CKPT)


# Compatibility for final demo bundle writer.
def save_checkpoint_atomic(state, destination):
    checkpoint_atomic(state, Path(destination))


In [ ]:

# CELL: FIND_BEST_CHECKPOINT

from pathlib import Path
import os

print("=" * 60)
print("CHECKING SAVED MODEL")
print("=" * 60)

ROOT = Path("/content/drive/MyDrive")

EXPECTED = (
    ROOT
    / "BraTS2023_GLI"
    / "runs_v2"
    / "unet_v2_resumable_001"
    / "checkpoints"
    / "unet3d_brats_best.pth"
)

print("Expected:", EXPECTED)
print("Exists:", EXPECTED.is_file())

found = []

if EXPECTED.is_file():
    found.append(EXPECTED)

else:
    print("\nSearching MyDrive...")

    for root, dirs, files in os.walk(ROOT):

        # Bo qua cac thu muc dataset lon
        dirs[:] = [
            d for d in dirs
            if not d.startswith("processed_npz")
            and d not in (
                "raw",
                "brats2023_local_npz",
                ".Trash"
            )
        ]

        if "unet3d_brats_best.pth" in files:
            path = Path(root) / "unet3d_brats_best.pth"
            found.append(path)

print("\n" + "=" * 60)
print("RESULT")
print("=" * 60)

for path in found:
    print("FOUND:", path)
    print("Size MB:", round(
        path.stat().st_size / (1024 ** 2), 2
    ))

if len(found) == 1:
    BEST_CKPT = found[0]
    print("\nBEST CHECKPOINT FOUND!")
    print("Path:", BEST_CKPT)

elif len(found) > 1:
    print("\nMultiple checkpoints found.")
    print("Can xac minh checkpoint dung.")

else:
    print("\nCheckpoint chua hien trong MyDrive.")
    print("Hay tao shortcut toi thu muc checkpoints")
    print("trong MyDrive roi mount lai Drive.")


## 8. Đồ thị hội tụ và Bảng tổng kết kết quả

In [ ]:
if not globals().get('RUN_COMPLETE', False):
    print('SKIPPED: Đồ thị huấn luyện — training chưa hoàn thành. Không có dữ liệu giả.')
else:
    # 1. Nạp history trực tiếp từ file checkpoint đã lưu trên Drive/Local
    # Complete run history includes epochs after the best checkpoint.
    history = best_run['history']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Đồ thị Training Loss
    axes[0].plot(
        range(1, len(history["train_loss"]) + 1),
        history["train_loss"],
        label="Train loss",
        color="#1f77b4",
        lw=2,
    )
    axes[0].set_title("Training Convergence (Loss)", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, linestyle="--", alpha=0.5)
    axes[0].legend()

    # Đồ thị Validation Dice
    axes[1].plot(
        history["fast_val_epochs"],
        history["fast_val_dice"],
        label="Fast validation Dice",
        color="#2ca02c",
        marker="o",
    )
    axes[1].plot(
        history["full_val_epochs"],
        history["full_val_dice"],
        label="Full validation Dice",
        color="#d62728",
        marker="s",
        lw=2,
    )
    axes[1].set_title("Validation Dice Score", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice Score")
    axes[1].grid(True, linestyle="--", alpha=0.5)
    axes[1].legend()

    plt.tight_layout()
    fig.savefig(reports_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if not globals().get('RUN_COMPLETE', False):
    print('SKIPPED: Bảng tổng kết — training chưa hoàn thành. Không có dữ liệu giả.')
else:
    print(
        f"{'MODEL':<14} | {'SPLIT':<10} | {'DICE':>7} | {'WT':>7} | {'TC':>7} | "
        f"{'ET':>7} | {'IOU':>7} | {'HD95 MM':>9} | {'BEST EPOCH':>10}"
    )
    print("=" * 116)
    for model_name, run in MODEL_RUNS.items():
        for split_name in ("val", "test"):
            if f"{split_name}_summary" not in run:
                continue
            summary = run[f"{split_name}_summary"]
            print(
                f"{model_name:<14} | {split_name:<10} | {summary['dice_mean']:>7.4f} | "
                f"{summary['dice_wt']:>7.4f} | {summary['dice_tc']:>7.4f} | "
                f"{summary['dice_et']:>7.4f} | {summary['iou_mean']:>7.4f} | "
                f"{summary['hd95_mean']:>9.3f} | {run['best_epoch']:>10}"
            )
    print("=" * 116)
    print("Model selection used validation only; test metrics are final reporting metrics.")


## 9. Trực quan hóa kết quả phân đoạn 3D đa mặt phẳng

In [ ]:
def visualize_slices(
    image_4d: np.ndarray,
    prediction: np.ndarray,
    case_id: str,
    model_name: str,
    ground_truth: Optional[np.ndarray] = None,
):
    flair = image_4d[3]
    pred = np.asarray(prediction).squeeze()
    gt = None if ground_truth is None else np.asarray(ground_truth).squeeze()
    selection_mask = gt if gt is not None and np.any(gt > 0) else pred

    z_sum = np.sum(selection_mask > 0, axis=(0, 1))
    y_sum = np.sum(selection_mask > 0, axis=(0, 2))
    x_sum = np.sum(selection_mask > 0, axis=(1, 2))
    z_idx = int(np.argmax(z_sum)) if z_sum.max() > 0 else flair.shape[2] // 2
    y_idx = int(np.argmax(y_sum)) if y_sum.max() > 0 else flair.shape[1] // 2
    x_idx = int(np.argmax(x_sum)) if x_sum.max() > 0 else flair.shape[0] // 2

    from matplotlib.colors import ListedColormap

    segmentation_cmap = ListedColormap(["none", "#e41a1c", "#4daf4a", "#ff7f00"])
    planes = [
        ("Axial", flair[:, :, z_idx], None if gt is None else gt[:, :, z_idx], pred[:, :, z_idx]),
        ("Coronal", flair[:, y_idx, :], None if gt is None else gt[:, y_idx, :], pred[:, y_idx, :]),
        ("Sagittal", flair[x_idx, :, :], None if gt is None else gt[x_idx, :, :], pred[x_idx, :, :]),
    ]
    columns = 3 if gt is not None else 2
    fig, axes = plt.subplots(3, columns, figsize=(5 * columns, 12))
    for row, (plane_name, base, target_mask, pred_mask) in enumerate(planes):
        axes[row, 0].imshow(base.T, cmap="gray", origin="lower")
        axes[row, 0].set_title(f"{plane_name} - FLAIR")
        axes[row, 0].axis("off")
        pred_column = 2 if gt is not None else 1
        if gt is not None:
            axes[row, 1].imshow(base.T, cmap="gray", origin="lower")
            axes[row, 1].imshow(target_mask.T, cmap=segmentation_cmap, vmin=0, vmax=3, alpha=0.5, origin="lower")
            axes[row, 1].set_title(f"{plane_name} - Ground truth")
            axes[row, 1].axis("off")
        axes[row, pred_column].imshow(base.T, cmap="gray", origin="lower")
        axes[row, pred_column].imshow(pred_mask.T, cmap=segmentation_cmap, vmin=0, vmax=3, alpha=0.5, origin="lower")
        axes[row, pred_column].set_title(f"{plane_name} - {model_name}")
        axes[row, pred_column].axis("off")
    plt.suptitle(f"3D segmentation - {case_id}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
if not globals().get('RUN_COMPLETE', False):
    print('SKIPPED: Trực quan hóa — training chưa hoàn thành. Không có dữ liệu giả.')
else:
    # Demonstrate the selected checkpoint on one held-out test case.
    sample_batch = next(iter(test_loader))
    sample_image = sample_batch["image"].cpu()
    sample_label = sample_batch["label"]
    sample_case_id = sample_batch["case_id"][0]
    sample_spacing = tuple(float(value) for value in sample_batch["spacing"][0].cpu().numpy())
    sample_affine = sample_batch["affine"][0].cpu().numpy()

    sample_prediction_tensor = predict_volume(best_model, sample_image)
    sample_prediction = sample_prediction_tensor[0, 0].cpu().numpy().astype(np.uint8)
    sample_metrics = compute_case_metrics(
        sample_prediction,
        sample_label[0, 0],
        sample_spacing,
        include_surface=True,
    )

    prediction_path = predictions_dir / f"{sample_case_id}_{best_model_name}_seg.nii.gz"
    prediction_image = nib.Nifti1Image(sample_prediction, sample_affine)
    prediction_image.header.set_xyzt_units('mm')
    prediction_image.header['descrip'] = b'Internal labels: 0 background, 1 NCR/NET, 2 ED, 3 ET'
    nib.save(prediction_image, prediction_path)
    print(f"Prediction saved with original affine: {prediction_path}")

    visualize_slices(
        sample_image[0].cpu().numpy(),
        sample_prediction,
        sample_case_id,
        best_model_name,
        ground_truth=sample_label[0].numpy(),
    )


## 10. Research segmentation report

Bao cao nay trinh bay output dinh luong cua model. No khong dua ra chan doan,
khong de xuat dieu tri va khong duoc dung cho quyet dinh lam sang.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

def save_segmentation_figure(
    image_4d: np.ndarray,
    prediction: np.ndarray,
    output_file: Union[str, Path],
    ground_truth: Optional[np.ndarray] = None,
) -> Path:
    from matplotlib.colors import ListedColormap

    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    flair = np.asarray(image_4d)[3]
    pred = np.asarray(prediction).squeeze()
    gt = None if ground_truth is None else np.asarray(ground_truth).squeeze()

    selection_mask = gt if gt is not None and np.any(gt > 0) else pred

    z_sum = np.sum(selection_mask > 0, axis=(0, 1))
    y_sum = np.sum(selection_mask > 0, axis=(0, 2))
    x_sum = np.sum(selection_mask > 0, axis=(1, 2))

    z_idx = int(np.argmax(z_sum)) if z_sum.max() > 0 else flair.shape[2] // 2
    y_idx = int(np.argmax(y_sum)) if y_sum.max() > 0 else flair.shape[1] // 2
    x_idx = int(np.argmax(x_sum)) if x_sum.max() > 0 else flair.shape[0] // 2

    segmentation_cmap = ListedColormap(
        ["none", "#e41a1c", "#4daf4a", "#ff7f00"]
    )

    planes = [
        ("Axial", flair[:, :, z_idx], None if gt is None else gt[:, :, z_idx], pred[:, :, z_idx]),
        ("Coronal", flair[:, y_idx, :], None if gt is None else gt[:, y_idx, :], pred[:, y_idx, :]),
        ("Sagittal", flair[x_idx, :, :], None if gt is None else gt[x_idx, :, :], pred[x_idx, :, :]),
    ]

    columns = 3 if gt is not None else 2
    figure, axes = plt.subplots(3, columns, figsize=(12, 12))

    if columns == 2:
        axes = np.asarray(axes).reshape(3, 2)

    for row, (plane_name, base, target_mask, pred_mask) in enumerate(planes):
        axes[row, 0].imshow(base.T, cmap="gray", origin="lower")
        axes[row, 0].set_title(f"{plane_name} - FLAIR")
        axes[row, 0].axis("off")

        prediction_column = 2 if gt is not None else 1

        if gt is not None:
            axes[row, 1].imshow(base.T, cmap="gray", origin="lower")
            axes[row, 1].imshow(
                target_mask.T,
                cmap=segmentation_cmap,
                vmin=0,
                vmax=3,
                alpha=0.5,
                origin="lower",
            )
            axes[row, 1].set_title(f"{plane_name} - Ground truth")
            axes[row, 1].axis("off")

        axes[row, prediction_column].imshow(
            base.T,
            cmap="gray",
            origin="lower",
        )
        axes[row, prediction_column].imshow(
            pred_mask.T,
            cmap=segmentation_cmap,
            vmin=0,
            vmax=3,
            alpha=0.5,
            origin="lower",
        )
        axes[row, prediction_column].set_title(
            f"{plane_name} - Model prediction"
        )
        axes[row, prediction_column].axis("off")

    legend_handles = [
        Patch(
            facecolor="#e41a1c",
            label="Label 1 - Necrotic/non-enhancing tumor",
        ),
        Patch(
            facecolor="#4daf4a",
            label="Label 2 - Edema",
        ),
        Patch(
            facecolor="#ff7f00",
            label="Label 3 - Enhancing tumor",
        ),
    ]

    figure.suptitle(
        "3D Brain Tumor Segmentation",
        fontsize=14,
        fontweight="bold",
    )

    figure.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=1,
        fontsize=9,
        frameon=True,
        bbox_to_anchor=(0.5, 0.01),
    )

    figure.tight_layout(rect=[0, 0.12, 1, 0.96])
    figure.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(figure)

    return output_path

In [ ]:
from fpdf import FPDF


class ResearchSegmentationReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 14)
        self.cell(
            0,
            8,
            "BRAIN TUMOR SEGMENTATION RESEARCH REPORT",
            new_x="LMARGIN",
            new_y="NEXT",
            align="C",
        )
        self.set_font("Helvetica", "I", 8)
        self.cell(
            0,
            5,
            "Research use only - not for clinical diagnosis or treatment",
            new_x="LMARGIN",
            new_y="NEXT",
            align="C",
        )
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(
            0,
            10,
            f"Page {self.page_no()} | SIC Capstone 2026",
            align="C",
        )


def export_research_report(
    case_id: str,
    model_name: str,
    spacing: Tuple[float, float, float],
    volumes: Dict[str, float],
    output_file: Union[str, Path],
    metrics: Optional[Dict[str, float]] = None,
    segmentation_image: Optional[Union[str, Path]] = None,
) -> Path:
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    pdf = ResearchSegmentationReport()
    pdf.set_auto_page_break(auto=True, margin=20)
    pdf.add_page()

    # Section 1: Case and model
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(
        0,
        7,
        "1. CASE AND MODEL",
        new_x="LMARGIN",
        new_y="NEXT",
    )

    pdf.set_font("Helvetica", "", 9)
    pdf.cell(
        0,
        5,
        f"Case ID: {case_id}",
        new_x="LMARGIN",
        new_y="NEXT",
    )
    pdf.cell(
        0,
        5,
        f"Model: {model_name}",
        new_x="LMARGIN",
        new_y="NEXT",
    )
    pdf.cell(
        0,
        5,
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        new_x="LMARGIN",
        new_y="NEXT",
    )
    pdf.cell(
        0,
        5,
        (
            f"Voxel spacing: {spacing[0]:.4f} x "
            f"{spacing[1]:.4f} x {spacing[2]:.4f} mm"
        ),
        new_x="LMARGIN",
        new_y="NEXT",
    )
    pdf.ln(3)

    # Section 2: Predicted volumes
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(
        0,
        7,
        "2. PREDICTED VOLUMES",
        new_x="LMARGIN",
        new_y="NEXT",
    )

    pdf.set_font("Helvetica", "B", 9)
    pdf.cell(90, 6, "Region", border=1, align="C")
    pdf.cell(
        100,
        6,
        "Predicted volume (cm3)",
        border=1,
        align="C",
        new_x="LMARGIN",
        new_y="NEXT",
    )

    pdf.set_font("Helvetica", "", 9)
    for region_name, display_name in (
        ("wt", "Whole Tumor"),
        ("tc", "Tumor Core"),
        ("et", "Enhancing Tumor"),
    ):
        pdf.cell(90, 6, display_name, border=1)
        pdf.cell(
            100,
            6,
            f"{volumes[region_name]:.3f}",
            border=1,
            align="C",
            new_x="LMARGIN",
            new_y="NEXT",
        )

    pdf.ln(3)

    has_segmentation_image = segmentation_image is not None

    if has_segmentation_image:
        image_path = Path(segmentation_image)

        if not image_path.is_file():
            raise FileNotFoundError(
                f"Segmentation image not found: {image_path}"
            )

        # Section 3: Segmentation visualization
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 11)
        pdf.cell(
            0,
            7,
            "3. MODEL SEGMENTATION VISUALIZATION",
            new_x="LMARGIN",
            new_y="NEXT",
        )

        pdf.set_font("Helvetica", "", 8)
        pdf.multi_cell(
            0,
            4,
            "FLAIR images with ground-truth and model-predicted "
            "segmentation overlays. Research visualization only.",
        )
        pdf.ln(3)

        pdf.image(
            str(image_path),
            x=15,
            y=45,
            w=180,
        )

    # Section 4 or 3: Reference-based metrics
    if metrics is not None:
        metrics_section = 4 if has_segmentation_image else 3

        pdf.add_page()
        pdf.set_font("Helvetica", "B", 11)
        pdf.cell(
            0,
            7,
            f"{metrics_section}. REFERENCE-BASED METRICS",
            new_x="LMARGIN",
            new_y="NEXT",
        )

        pdf.set_font("Helvetica", "", 8)
        pdf.multi_cell(
            0,
            4,
            "These metrics require a ground-truth mask and are "
            "evaluation results, not confidence scores. Dice is computed per region; "
            "HD95 uses pooled bidirectional surface distances (26-neighbor boundary), "
            "zero for both-empty masks and the volume diagonal for a single empty mask. "
            "This is a custom metric policy, not an official BraTS challenge evaluation.",
        )

        pdf.set_font("Helvetica", "B", 8)
        for title, width in (
            ("Region", 40),
            ("Dice", 30),
            ("IoU", 30),
            ("Precision", 30),
            ("Recall", 30),
            ("HD95 mm", 30),
        ):
            pdf.cell(width, 6, title, border=1, align="C")

        pdf.ln()
        pdf.set_font("Helvetica", "", 8)

        for region_name, display_name in (
            ("wt", "WT"),
            ("tc", "TC"),
            ("et", "ET"),
        ):
            values = [
                display_name,
                f"{metrics[f'dice_{region_name}']:.4f}",
                f"{metrics[f'iou_{region_name}']:.4f}",
                f"{metrics[f'precision_{region_name}']:.4f}",
                f"{metrics[f'recall_{region_name}']:.4f}",
                f"{metrics[f'hd95_{region_name}']:.3f}",
            ]

            for value, width in zip(
                values,
                (40, 30, 30, 30, 30, 30),
            ):
                pdf.cell(width, 6, value, border=1, align="C")

            pdf.ln()

    # Section limitations
    if metrics is not None:
        limitations_section = 5 if has_segmentation_image else 4
    else:
        limitations_section = 4 if has_segmentation_image else 3

    pdf.add_page()
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(
        0,
        7,
        f"{limitations_section}. LIMITATIONS",
        new_x="LMARGIN",
        new_y="NEXT",
    )

    pdf.set_font("Helvetica", "", 9)
    pdf.multi_cell(
        0,
        5,
        "This automated segmentation is a research output. It has not "
        "been clinically validated, does not constitute a diagnosis, "
        "and must not be used for treatment or radiotherapy planning.",
    )

    pdf.output(str(output_path))
    print(f"Research report saved: {output_path}")

    return output_path

In [ ]:
if not globals().get('RUN_COMPLETE', False):
    print('SKIPPED: Báo cáo/PDF và demo bundle — training chưa hoàn thành. Không có dữ liệu giả.')
else:
    sample_volumes = {
        region: sample_metrics[f"volume_cm3_{region}"]
        for region in REGION_BUILDERS
    }

    # Save the prediction visualization as an image
    segmentation_figure_path = reports_dir / (
        f"{sample_case_id}_{best_model_name}_segmentation.png"
    )

    save_segmentation_figure(
        image_4d=sample_image[0].cpu().numpy(),
        prediction=sample_prediction,
        ground_truth=sample_label[0].numpy(),
        output_file=segmentation_figure_path,
    )

    print(f"Segmentation figure saved: {segmentation_figure_path}")
    print(f"Figure exists: {segmentation_figure_path.exists()}")

    # Export PDF with the segmentation image
    sample_report_path = reports_dir / (
        f"{sample_case_id}_{best_model_name}_research_report.pdf"
    )

    export_research_report(
        case_id=sample_case_id,
        model_name=best_model_name,
        spacing=sample_spacing,
        volumes=sample_volumes,
        metrics=sample_metrics,
        output_file=sample_report_path,
        segmentation_image=segmentation_figure_path,
    )

    # Export inference-only weights + the exact model/preprocessing/inference contract.
    # This bundle is separate from optimizer checkpoints; training remains resumable.
    demo_weights_path = export_dir / 'unet_v1_demo.pth'
    demo_metadata = {
        'format_version': 1, 'model_name': best_model_name, 'best_epoch': best_run['best_epoch'],
        'best_state_id': best_checkpoint['state_id'], 'protocol': best_run['protocol'],
        'architecture': {'spatial_dims': 3, 'in_channels': 4, 'out_channels': 4,
                         **best_run['protocol']['architecture']},
        'preprocessing': {'modalities': list(MODALITIES), 'clip_percentiles': [0.5, 99.5],
                          'foreground': 'intensity > 0', 'normalize': 'per-modality foreground z-score',
                          'background': 0, 'resample': False, 'crop': False,
                          'raw_label_schema': RAW_LABEL_SCHEMA},
        'labels': {'0': 'background', '1': 'NCR', '2': 'ED', '3': 'ET'},
        'validation': best_run['val_summary'], 'test': best_run['test_summary'],
    }
    save_checkpoint_atomic({'model_state': {k: v.detach().cpu() for k, v in best_model.state_dict().items()},
                            'metadata': demo_metadata, 'protocol': best_run['protocol']}, demo_weights_path)
    digest = hashlib.sha256()
    with demo_weights_path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    demo_metadata['weights_sha256'] = digest.hexdigest()
    write_json_atomic(demo_metadata, export_dir / 'model_metadata.json')
    shutil.copy2(SPLIT_MANIFEST_PATH, export_dir / 'split_manifest.json')
    shutil.copy2(reports_dir / 'unet3d_evaluation.json', export_dir / 'evaluation.json')
    shutil.copy2(reports_dir / 'training_curves.png', export_dir / 'training_curves.png')
    shutil.copy2(sample_report_path, export_dir / sample_report_path.name)
    shutil.copy2(prediction_path, export_dir / prediction_path.name)

    # A small inference-only runner makes the bundle usable without this notebook.
    runner_source = """from pathlib import Path
    import argparse, json
    import nibabel as nib
    import numpy as np
    import torch
    from monai.networks.nets import UNet
    from monai.inferers import sliding_window_inference
    from scipy.ndimage import label

    def main():
        parser = argparse.ArgumentParser(description='3D U-Net v1: four aligned BraTS MRI NIfTI files')
        for name in ('t1n', 't1c', 't2w', 't2f'):
            parser.add_argument('--' + name, required=True)
        parser.add_argument('--output', default='prediction.nii.gz')
        args = parser.parse_args()
        root = Path(__file__).resolve().parent
        package = torch.load(root / 'unet_v1_demo.pth', map_location='cpu', weights_only=True)
        metadata = package['metadata']
        architecture = dict(metadata['architecture'])
        architecture['channels'] = tuple(architecture['channels'])
        architecture['strides'] = tuple(architecture['strides'])
        model = UNet(**architecture)
        model.load_state_dict(package['model_state'], strict=True)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.to(device).eval()
        reference, channels = None, []
        for name in metadata['preprocessing']['modalities']:
            nii = nib.load(getattr(args, name))
            array = np.asarray(nii.dataobj, dtype=np.float32)
            if array.ndim != 3 or not np.isfinite(array).all():
                raise ValueError('MRI must be a finite 3D volume: ' + name)
            if reference is None:
                reference = nii
            elif array.shape != reference.shape or not np.allclose(nii.affine, reference.affine, rtol=0, atol=1e-4):
                raise ValueError('All modalities must have the same shape and affine')
            mask = array > 0
            if not mask.any():
                raise ValueError('Empty MRI signal: ' + name)
            low, high = np.percentile(array[mask], [0.5, 99.5])
            array = np.clip(array, low, high)
            mean, std = array[mask].mean(), array[mask].std()
            if std <= 1e-6:
                raise ValueError('Near-zero MRI intensity variance: ' + name)
            array = (array - mean) / std
            array[~mask] = 0
            channels.append(array.astype(np.float32, copy=False))
        image = torch.from_numpy(np.stack(channels)).unsqueeze(0)
        config = metadata['protocol']
        def predict(patches):
            with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
                return model(patches).float()
        def infer(volume):
            return sliding_window_inference(volume, tuple(config['patch_size']), 1, predict,
                overlap=config['inference']['sw_overlap'], mode='gaussian',
                sw_device=device, device=torch.device('cpu')).softmax(dim=1)
        with torch.inference_mode():
            probabilities = infer(image)
            if config['inference']['tta_axis'] is not None:
                probabilities = (probabilities + infer(image.flip(2)).flip(2)) / 2
            prediction = probabilities.argmax(1)[0].numpy().astype(np.uint8)
        for region in (1, 2, 3):
            components, _ = label(prediction == region)
            small = np.bincount(components.ravel()) < config['inference']['small_objects_voxels']
            small[0] = False
            prediction[small[components]] = 0
        result = nib.Nifti1Image(prediction, reference.affine)
        result.header.set_xyzt_units('mm')
        output = Path(args.output)
        output.parent.mkdir(parents=True, exist_ok=True)
        nib.save(result, output)
        spacing = nib.affines.voxel_sizes(reference.affine)
        volumes = {name: float(mask.sum() * np.prod(spacing) / 1000)
            for name, mask in [('WT', prediction > 0), ('TC', np.isin(prediction, [1, 3])), ('ET', prediction == 3)]}
        print(json.dumps({'output': str(output), 'volumes_cm3': volumes}, indent=2))

    if __name__ == '__main__':
        main()
    """
    (export_dir / 'predict.py').write_text(runner_source, encoding='utf-8')
    (export_dir / 'requirements.txt').write_text(
        f"torch=={str(torch.__version__).split('+')[0]}\nmonai=={__import__('monai').__version__}\nnibabel>=5.2,<6\nnumpy>=1.24,<3\nscipy>=1.11,<2\n",
        encoding='utf-8')
    (export_dir / 'README.txt').write_text(
        '3D U-Net v1 demo bundle. Research use only.\n'
        'Install: pip install -r requirements.txt\n'
        'Run: python predict.py --t1n case-t1n.nii.gz --t1c case-t1c.nii.gz '
        '--t2w case-t2w.nii.gz --t2f case-t2f.nii.gz --output prediction.nii.gz\n'
        'Inputs must be pre-aligned 3D BraTS MRI on the same grid. Output labels: 0 BG, 1 NCR, 2 ED, 3 ET.\n'
        'Use model_metadata.json to match preprocessing and inference when connecting to a web demo.\n', encoding='utf-8')
    bundle_path = run_dir / 'unet_v1_demo.zip'
    with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for item in sorted(export_dir.iterdir()):
            archive.write(item, arcname=item.name)
    print(f'Demo weights: {demo_weights_path}')
    print(f'Download this bundle after training: {bundle_path}')
    if IN_COLAB:
        from IPython.display import display, FileLink
        display(FileLink(str(bundle_path)))
